In [1]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import torch
from genkit.datasets import fetch_synthetic_data
from genkit.flow_matching import GaussianFlowLinear
from genkit.metrics import mmd_rbf, tail_coverage_error
from genkit.nn import MLPModel
from genkit.training import train


/home/hcherkaoui/src/flowbench/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", force=True)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print(f"[INFO] device={device} dtype={dtype}")

[INFO] device=cpu dtype=torch.float32


In [3]:
class _HeteroscedasticNet(torch.nn.Module):
    def __init__(self, field, dim, n_noise_levels, gate_width, init_log_var, learn_scales):
        super().__init__()
        self.field = field
        self.noise_gate = torch.nn.Sequential(
            torch.nn.Linear(dim, gate_width),
            torch.nn.SiLU(),
            torch.nn.Linear(gate_width, n_noise_levels),
        )
        if learn_scales:
            self.log_var_multipliers = torch.nn.Parameter(init_log_var.clone())
        else:
            self.register_buffer("log_var_multipliers", init_log_var.clone())

    def forward(self, x, t):
        return self.field(x, t)


class HeteroscedasticGaussianFlowLinear(GaussianFlowLinear):
    def __init__(
        self,
        *args,
        n_noise_levels=4,
        init_var_multipliers=None,
        gate_width=64,
        learn_scales=True,
        temperature=1.0,
        scale_reg=1e-4,
        entropy_reg=0.0,
        ema_decay=0.99,
        min_log_var=-6.0,
        max_log_var=6.0,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self._n_noise_levels = int(n_noise_levels)
        self._temperature = float(temperature)
        self._scale_reg = float(scale_reg)
        self._entropy_reg = float(entropy_reg)
        self._ema_decay = float(ema_decay)
        self._min_log_var = float(min_log_var)
        self._max_log_var = float(max_log_var)
        init_var = torch.logspace(-2.0, 2.0, self._n_noise_levels, base=2.0) if init_var_multipliers is None else torch.as_tensor(init_var_multipliers)
        init_log_var = init_var.to(device=self._device, dtype=self._fdtype).log().clamp(self._min_log_var, self._max_log_var)
        self._init_log_var_multipliers = init_log_var.detach().clone()
        self._net = _HeteroscedasticNet(self._net, self._dim, self._n_noise_levels, gate_width, init_log_var, learn_scales)
        self._net.to(device=self._device, dtype=self._fdtype)
        self._noise_probs = torch.full((self._n_noise_levels,), 1.0 / self._n_noise_levels, device=self._device, dtype=self._fdtype)
        self._noise_probs_ema = self._noise_probs.clone()
        self._last_entropy_penalty = torch.zeros((), device=self._device, dtype=self._fdtype)

    def _var_multipliers(self):
        return self._net.log_var_multipliers.clamp(self._min_log_var, self._max_log_var).exp()

    def _precompute_loss(self, x, z=None, t=None):
        if z is not None:
            raise ValueError("This prototype samples x0 from the data-dependent source and does not accept z.")
        x_1 = x.to(device=self._device, dtype=self._fdtype)
        if tuple(x_1.shape[1:]) != self._sample_shape:
            raise ValueError(f"Expected x1 shape (N, *{self._sample_shape}), got {tuple(x_1.shape)}")
        batch_size = x_1.shape[0]
        t = self._check_t(t, batch_size)
        logits = self._net.noise_gate(x_1.reshape(batch_size, -1))
        pi = torch.softmax(logits, dim=-1)
        y = torch.nn.functional.gumbel_softmax(logits, tau=self._temperature, hard=True, dim=-1)
        var_mult = (y * self._var_multipliers()).sum(dim=-1)
        eps = torch.randn_like(x_1)
        x_0 = self._sigma_max * self._expand_batch_scalar(torch.sqrt(var_mult.clamp_min(self._eps)), x_1) * eps
        batch_probs = pi.detach().mean(dim=0)
        self._noise_probs.copy_(batch_probs)
        self._noise_probs_ema.mul_(self._ema_decay).add_((1.0 - self._ema_decay) * batch_probs)
        self._noise_probs_ema.div_(self._noise_probs_ema.sum().clamp_min(1e-12))
        self._last_entropy_penalty = -(pi * pi.clamp_min(self._eps).log()).sum(dim=-1).mean()
        t_data = self._expand_batch_scalar(t, x_1)
        x_t = (1.0 - t_data) * x_0 + t_data * x_1
        v_t = x_1 - x_0
        v_t_hat = self._net(x_t, t)
        return v_t_hat, v_t, t

    def _sample_source_default(self, n_samples):
        probs = self._noise_probs_ema / self._noise_probs_ema.sum().clamp_min(1e-12)
        idx = torch.multinomial(probs, int(n_samples), replacement=True)
        var_mult = self._var_multipliers().index_select(0, idx)
        eps = torch.randn(n_samples, *self._sample_shape, device=self._device, dtype=self._fdtype)
        return self._sigma_max * self._expand_batch_scalar(torch.sqrt(var_mult), eps) * eps

    def loss(self, x, z=None, t=None):
        mse_loss = self._reduce(self._loss(x=x, z=z, t=t))
        scale_penalty = (self._net.log_var_multipliers - self._init_log_var_multipliers).pow(2).mean()
        return mse_loss + self._scale_reg * scale_penalty + self._entropy_reg * self._last_entropy_penalty

    @property
    def noise_var_multipliers(self):
        return self._var_multipliers().detach().cpu()

    @property
    def noise_std_multipliers(self):
        return torch.sqrt(self._var_multipliers()).detach().cpu()

    @property
    def noise_probs(self):
        return self._noise_probs.detach().cpu()

    @property
    def noise_probs_ema(self):
        return self._noise_probs_ema.detach().cpu()

In [4]:
alpha = 1.7
print(f"[INFO] loading isotropic alpha-stable data alpha={alpha}")
x_train, _, x_test = fetch_synthetic_data(
    "alpha_stable",
    alpha=alpha,
    dim=2,
    n_samples=170_000,
    val_size=0.001,
    test_size=0.8,
    standardize=False,
    device=device,
    dtype=dtype,
)
n_train = x_train.shape[0]
n_test = x_test.shape[0]
print(f"[INFO] train_shape={n_train} test_shape={n_test}")


[INFO] loading isotropic alpha-stable data alpha=1.7
[INFO] train_shape=33829 test_shape=136000


In [5]:
def make_net():
    return MLPModel(dim=2, width=128, depth=3).to(device=device, dtype=dtype)


def make_models(seed):
    torch.manual_seed(seed)
    print(f"[INFO] building models seed={seed}")
    baseline = GaussianFlowLinear(net=make_net(), dim=2, n_steps=128, sigma_max=1.0, device=device)
    hetero = HeteroscedasticGaussianFlowLinear(
        net=make_net(),
        dim=2,
        n_steps=128,
        n_noise_levels=4,
        sigma_max=1.0,
        gate_width=64,
        device=device,
    )
    return baseline, hetero


tce_probs = {
    "TCE(90)": 1e-1,
    "TCE(99)": 1e-2,
    "TCE(99,9)": 1e-3,
    "TCE(99,99)": 1e-4,
}


@torch.no_grad()
def evaluate_model(name, model, trial, x_ref, n_tail=50_000, n_mmd=2_000):
    print(f"[INFO] evaluating trial={trial} model={name} n_tail={n_tail} n_mmd={n_mmd}")
    x_ref = x_ref[:n_tail].detach().cpu().to(torch.float64)
    x_gen = model.sample(len(x_ref)).detach().cpu().to(torch.float64)
    row = {
        "trial": trial,
        "model": name,
        "mmd_rbf": mmd_rbf(x_ref[:n_mmd], x_gen[:n_mmd]),
    }
    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_ref, x_gen, probs=probs, tail="upper", mode="log")
    return row


In [6]:
n_tail = n_test
n_mmd = n_test // 10
n_trials = 5
train_kwargs = dict(batch_size=128, n_epochs=32, lr=5e-4, device=device, use_adamw=False, lr_schedule="constant", freq_logging=8)

In [7]:
print(f"[INFO] n_trials={n_trials} train_kwargs={train_kwargs}")

rows = []
for trial in range(1, n_trials + 1):

    print(f"[INFO] trial {trial}/{n_trials}: start")
    baseline, hetero = make_models(seed=trial - 1)

    print(f"[INFO] trial {trial}/{n_trials}: train GF linear")
    baseline, _ = train(baseline, x_train, **train_kwargs)

    print(f"[INFO] trial {trial}/{n_trials}: train heteroscedastic GF linear")
    hetero, _ = train(hetero, x_train, **train_kwargs)

    print(f"[INFO] trial {trial}/{n_trials}: evaluate GF linear")
    rows.append(evaluate_model("GF linear", baseline, trial, x_test, n_tail=n_tail, n_mmd=n_mmd))

    print(f"[INFO] trial {trial}/{n_trials}: evaluate heteroscedastic GF linear")
    rows.append(evaluate_model("heteroscedastic GF linear", hetero, trial, x_test, n_tail=n_tail, n_mmd=n_mmd))

    print(f"[INFO] trial {trial}/{n_trials}: done")


[INFO] train | epochs=32 bs=128 lr=0.0005 schedule=constant warmup_steps=0 cosine_eta_min_ratio=0 opt=Adam wd=0 device=cpu dtype=torch.float32 ckpt_dir=None


[INFO] n_trials=5 train_kwargs={'batch_size': 128, 'n_epochs': 32, 'lr': 0.0005, 'device': device(type='cpu'), 'use_adamw': False, 'lr_schedule': 'constant', 'freq_logging': 8}
[INFO] trial 1/5: start
[INFO] building models seed=0
[INFO] trial 1/5: train GF linear


[INFO] epoch   8/ 32 | loss 6.793106 ± 50.398438 | grad_var 2.202e-03 | grad_norm 1.593e+01 | lr 5.000e-04
[INFO] epoch  16/ 32 | loss 5.483262 ± 30.532242 | grad_var 7.867e-03 | grad_norm 3.011e+01 | lr 5.000e-04
[INFO] epoch  24/ 32 | loss 3.821721 ± 14.093604 | grad_var 1.617e-02 | grad_norm 4.317e+01 | lr 5.000e-04
[INFO] epoch  32/ 32 | loss 3.648600 ± 6.632852 | grad_var 1.780e-02 | grad_norm 4.530e+01 | lr 5.000e-04
[INFO] train | done
[INFO] train | epochs=32 bs=128 lr=0.0005 schedule=constant warmup_steps=0 cosine_eta_min_ratio=0 opt=Adam wd=0 device=cpu dtype=torch.float32 ckpt_dir=None


[INFO] trial 1/5: train heteroscedastic GF linear


[INFO] epoch   8/ 32 | loss 12.352037 ± 176.301132 | grad_var 2.956e-03 | grad_norm 1.850e+01 | lr 5.000e-04
[INFO] epoch  16/ 32 | loss 0.991713 ± 3.175726 | grad_var 6.278e-04 | grad_norm 8.524e+00 | lr 5.000e-04
[INFO] epoch  24/ 32 | loss 2.838167 ± 35.579048 | grad_var 7.961e-04 | grad_norm 9.598e+00 | lr 5.000e-04
[INFO] epoch  32/ 32 | loss 0.740186 ± 6.657307 | grad_var 2.324e-03 | grad_norm 1.640e+01 | lr 5.000e-04
[INFO] train | done


...

[INFO] trial 5/5: train GF linear


[INFO] epoch   8/ 32 | loss 8.746861 ± 63.333828 | grad_var 4.969e-04 | grad_norm 7.568e+00 | lr 5.000e-04
[INFO] epoch  16/ 32 | loss 4.289155 ± 15.017627 | grad_var 5.190e-03 | grad_norm 2.446e+01 | lr 5.000e-04
[INFO] epoch  24/ 32 | loss 5.759091 ± 25.340431 | grad_var 6.468e-02 | grad_norm 8.636e+01 | lr 5.000e-04
[INFO] epoch  32/ 32 | loss 3.696498 ± 6.410496 | grad_var 1.217e-02 | grad_norm 3.745e+01 | lr 5.000e-04
[INFO] train | done
[INFO] train | epochs=32 bs=128 lr=0.0005 schedule=constant warmup_steps=0 cosine_eta_min_ratio=0 opt=Adam wd=0 device=cpu dtype=torch.float32 ckpt_dir=None


[INFO] trial 5/5: train heteroscedastic GF linear


[INFO] epoch   8/ 32 | loss 6.502855 ± 82.621017 | grad_var 3.073e-04 | grad_norm 5.963e+00 | lr 5.000e-04
[INFO] epoch  16/ 32 | loss 1.668810 ± 13.181182 | grad_var 1.816e-04 | grad_norm 4.584e+00 | lr 5.000e-04
[INFO] epoch  24/ 32 | loss 1.340598 ± 11.706510 | grad_var 9.121e-03 | grad_norm 3.249e+01 | lr 5.000e-04
[INFO] epoch  32/ 32 | loss 0.721523 ± 3.129861 | grad_var 4.650e-04 | grad_norm 7.336e+00 | lr 5.000e-04
[INFO] train | done


[INFO] trial 5/5: evaluate GF linear
[INFO] evaluating trial=5 model=GF linear n_tail=136000 n_mmd=13600
[INFO] trial 5/5: evaluate heteroscedastic GF linear
[INFO] evaluating trial=5 model=heteroscedastic GF linear n_tail=136000 n_mmd=13600
[INFO] trial 5/5: done


In [8]:
print("[INFO] heteroscedastic mixture logic")

x_probe = x_train.to(device=device, dtype=dtype)

with torch.no_grad():
    logits = hetero._net.noise_gate(x_probe.reshape(x_probe.shape[0], -1))
    pi = torch.softmax(logits / hetero._temperature, dim=-1)

print("std multipliers:", hetero.noise_std_multipliers)
print("batch mean pi(x):", pi.mean(dim=0).cpu())
print("EMA generation probabilities:", hetero.noise_probs_ema)

[INFO] heteroscedastic mixture logic
std multipliers: tensor([0.0891, 0.7143, 1.2001, 1.8996])
batch mean pi(x): tensor([1.0000e+00, 1.8732e-07, 2.8234e-08, 2.7342e-08])
EMA generation probabilities: tensor([1.0000e+00, 1.9844e-07, 2.9860e-08, 2.8741e-08])


In [ ]:
print("[INFO] computing test-vs-test reference metrics")

x_ref = x_test.detach().cpu().to(torch.float64)

for trial in range(1, n_trials + 1):
    generator = torch.Generator(device="cpu").manual_seed(10_000 + trial)
    perm = torch.randperm(x_ref.shape[0], generator=generator)

    x_a = x_ref[perm[:n_test // 2]]
    x_b = x_ref[perm[n_test // 2: n_test]]

    row = {
        "trial": trial,
        "model": "test vs test",
        "mmd_rbf": mmd_rbf(x_a[:n_mmd], x_b[:n_mmd]),
    }

    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_a, x_b, probs=probs, tail="upper", mode="log")

    rows.append(row)

[INFO] computing test-vs-test reference metrics


In [ ]:
print("[INFO] aggregating results")
results = pd.DataFrame(rows)
tce_cols = ["TCE(90)", "TCE(99)", "TCE(99,9)", "TCE(99,99)"]
tce_x = [0.90, 0.99, 0.999, 0.9999]

grouped = results.groupby("model")
mean_results = grouped[["mmd_rbf", *tce_cols]].mean()
std_results = grouped[tce_cols].std().fillna(0.0)

fig, ax = plt.subplots(figsize=(6, 5))

for model_name, row in mean_results.iterrows():
    y = row[tce_cols].to_numpy(dtype=float)
    yerr = std_results.loc[model_name, tce_cols].to_numpy(dtype=float)
    lower = (y - yerr).clip(min=1e-12)
    upper = y + yerr
    label = f"{model_name} (MMD-RBF={row['mmd_rbf']:.4g})"
    line, = ax.plot(tce_x, y, marker="o", linewidth=2, label=label, alpha=0.5)
    ax.fill_between(tce_x, lower, upper, color=line.get_color(), alpha=0.2)

ax.set_xscale("logit")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-2)
ax.set_xticks(tce_x)
ax.set_xticklabels(["90", "99", "99.9", "99.99"])
ax.set_xlabel("tail quantile (%)", fontsize=12)
ax.set_ylabel("TCE, upper tail log error", fontsize=12)

ax.grid(True, which="both", alpha=0.4)
ax.legend()

plt.tight_layout()
